In [1]:
import pandas as pd
import numpy as np

from lightgbm import LGBMRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
data = pd.read_csv('Data/Final_Data.csv')

In [3]:
data = data.drop(columns=["SiteKey"])
data.shape

(1599443, 36)

In [ ]:
X = data[[
    "Day_sin","Day_cos",
    "Hour_sin","Hour_cos",
    "AirTemperature","ApparentTemperature",
    "RelativeHumidity","DewPointTemperature",
    "Temp_solar","Humidity_solar","Wind_solar",
    "WindSpeed","WindDir_sin","WindDir_cos",
    "SolarPotential",
]]

Y = data[[
    "Residual_now",
    "Residual_15min",
    "Residual_1h",
    "Residual_3h"
]]

Y_true = data[[
    "CapacityFactor",
    "Target_15min",
    "Target_1h",
    "Target_3h"
]]

n_splits = 5
split_size = int(len(X) / (n_splits + 1))
max_train_size = 1599443

mae_model = np.zeros(4)
mae_naive = np.zeros(4)
r2_scores = np.zeros(4)

for i in range(n_splits):
    train_end = split_size * (i+1)
    test_end = split_size * (i+2)
    train_start = max(0, train_end - max_train_size)

    X_train = X.iloc[train_start:train_end]
    X_test = X.iloc[train_end:test_end]

    Y_train = Y.iloc[train_start:train_end]
    true_test = Y_true.iloc[train_end:test_end]

    lag_tests = [
        data["Lag_1"].iloc[train_end:test_end].values,
        data["Lag_1"].iloc[train_end:test_end].values,
        data["Lag_2"].iloc[train_end:test_end].values,
        data["Lag_3"].iloc[train_end:test_end].values
    ]

    model = MultiOutputRegressor(
        LGBMRegressor(
            n_estimators=400,
            learning_rate=0.1,
            max_depth=18,
            num_leaves=256,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=20,
            random_state=42,
            n_jobs=-1,
            verbosity=-1,
        )
    )

    model.fit(X_train, Y_train)
    residual_pred = model.predict(X_test)

    pred = np.zeros_like(residual_pred)

    for h in range(4):
        pred[:, h] = lag_tests[h] + residual_pred[:, h]

    for h in range(4):
        mae_model[h] += mean_absolute_error(true_test.iloc[:, h], pred[:, h])
        mae_naive[h] += mean_absolute_error(true_test.iloc[:, h], lag_tests[h])
        r2_scores[h] += r2_score(true_test.iloc[:, h], pred[:, h])

mae_model /= n_splits
mae_naive /= n_splits
r2_scores /= n_splits

horizons = ["Now","15 min","1 hour","3 hour"]

print("\n=== WEATHER ONLY MODEL ===")
for i, h in enumerate(horizons):
    print(f"\n{h}")
    print("Naive MAE:", mae_naive[i])
    print("Model MAE:", mae_model[i])
    print("R2:", r2_scores[i])


=== WEATHER ONLY MODEL ===

Now
Naive MAE: 0.00891871835001148
Model MAE: 0.0058832046764588
R2: 0.9661672319494132

15 min
Naive MAE: 0.013283456971665514
Model MAE: 0.008539716037413003
R2: 0.9389879526667274

1 hour
Naive MAE: 0.025374984928098875
Model MAE: 0.016759432227951832
R2: 0.792475280274107

3 hour
Naive MAE: 0.04857314903658821
Model MAE: 0.02930443905068445
R2: 0.3790776108088877


In [21]:
horizons = ["Now","15 min","1 hour","3 hour"]

for i, h in enumerate(horizons):
    lgb_model = model.estimators_[i]
    importance = lgb_model.booster_.feature_importance(importance_type='gain')
    feat_imp = pd.DataFrame({
        "Feature": X.columns,
        "Importance": importance
    }).sort_values(by="Importance", ascending=False)
    
    print(f"\n=== TOP FEATURES: {h} ===")
    print(feat_imp.head(30))
    


=== TOP FEATURES: Now ===
                Feature  Importance
4        AirTemperature  204.894843
0               Day_sin  204.134234
7   DewPointTemperature  194.994685
5   ApparentTemperature  185.258952
14       SolarPotential  178.409945
6      RelativeHumidity  176.808890
8            Temp_solar  175.042486
9        Humidity_solar  168.416334
10           Wind_solar  145.188570
12          WindDir_sin  142.508476
1               Day_cos  140.455674
2              Hour_sin  115.722459
11            WindSpeed  114.402562
13          WindDir_cos  102.498942
3              Hour_cos   78.947098

=== TOP FEATURES: 15 min ===
                Feature  Importance
14       SolarPotential  396.267127
2              Hour_sin  374.734652
4        AirTemperature  363.272660
0               Day_sin  344.665818
7   DewPointTemperature  341.531652
8            Temp_solar  340.565639
9        Humidity_solar  333.403652
6      RelativeHumidity  313.900733
5   ApparentTemperature  302.176135
12     